# Lesson 10 - AI-agenter i produksjon

I denne leksjonen vil du lære **produksjonsmønstre** for AI-agenter ved bruk av Microsoft Agent Framework (Python). Vi dekker:

- **Observabilitet** — legge til timing og logging i agentinteraksjoner
- **Evaluering** — bruke en evaluatoragent for å score responskvalitet
- **Kostnadsstyring** — strategier for tokenoptimalisering og modellvalg

Scenarioet er en **reiseagent** som hjelper brukere med å planlegge turer, med overvåking og evaluering lagt på toppen.


## Oppsett


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Produksjonshensyn

Å flytte AI-agenter fra prototyper til produksjon krever nøye oppmerksomhet på tre søyler:

1. **Observabilitet** — Du trenger innsikt i hva agenten gjør, hvor lang tid det tar, og hvilke verktøy den kaller. Uten sporing og logging er det nesten umulig å feilsøke produksjonsproblemer.

2. **Evaluering** — Automatiserte kvalitetskontroller sikrer at agentens svar forblir nøyaktige, fullstendige og hjelpsomme over tid. En evalueringsagent kan gi poeng til svarene basert på definerte kriterier.

3. **Kostnadsstyring** — Token-bruk påvirker direkte kostnadene. Strategier som optimalisering av prompt, modellvalg og caching hjelper med å holde kostnadene under kontroll uten å ofre kvalitet.


## Bygge en observerbar agent

Vi definerer reiseverktøy og pakker agentkallet med tidtaking slik at vi kan overvåke latenstid. I produksjon ville du integrere med OpenTelemetry eller en lignende sporingstjeneste.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evalueringsmønstre

Et vanlig produksjonsmønster er å bruke en sekundær agent som en **vurderer**. Vurdereren vurderer hovedagentens svar mot forhåndsdefinerte kriterier som fullstendighet, nøyaktighet og hjelpsomhet.

Dette muliggjør:
- Automatiserte kvalitetsporter før svar når brukerne
- Regresjonsdeteksjon når forespørsler eller modeller endres
- Kontinuerlig overvåking av agentens ytelse over tid


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Kostnadsstyringsstrategier

Kontroll av kostnader er avgjørende for produksjons-AI-agenter. Her er nøkkelstrategier:

| Strategy | Description |
|---|---|
| **Promptoptimalisering** | Hold systeminstruksjoner konsise. Fjern overflødig kontekst for å redusere inntakstokener. |
| **Modellvalg** | Bruk mindre, billigere modeller (f.eks. GPT-4o-mini) for enkle oppgaver som klassifisering eller ekstraksjon, og reserver større modeller for komplekse resonnementer. |
| **Caching** | Bufre verktøyresultater og hyppige spørringer for å unngå unødvendige API-kall. |
| **Tokenbudsjetter** | Sett `max_tokens`-begrensninger for å forhindre uventet lange svar. |
| **Batching** | Grupper flere brukerforespørsler i ett API-kall der det er mulig. |

I praksis fungerer en trinnvis tilnærming godt: ruter enkle forespørsler til en rask, rimelig modell, og eskalerer kun komplekse spørringer til en mer kapabel modell.


## Sammendrag

I denne leksjonen lærte du hvordan du:

1. **Legger til observasjonsevne** til agentinteraksjoner med tidtaking og logging, og legger grunnlaget for sporing og overvåking.  
2. **Evaluerer agentresponser** automatisk ved hjelp av en evalueringsagent som vurderer fullstendighet, nøyaktighet og hjelpsomhet.  
3. **Håndterer kostnader** gjennom optimalisering av prompt, modellvalg, caching og tokenbudsjetter.  

Disse produksjonsmønstrene hjelper til med å sikre at AI-agentene dine er pålitelige, målbare og kostnadseffektive i stor skala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
